# Exercise 3, Python for data analysis

These exercises go with **Lecture 3 &mdash; Introduction to Python**. They practise the
three things that lecture introduced and that the rest of the course leans on: writing a
**class**, working with **NumPy** arrays, and opening gridded climate data with
**xarray**.

Fill only the cells marked

```python
# ==== YOUR CODE ====
```

Everything else is written for you.


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Exercise 1 &mdash; extend the `Station` class

Lecture 3 built a `Station` class holding a location and a growing list of daily mean
temperatures, with methods `add_day`, `mean` and `hot_days`. Here you add two methods.

**Your task.**
1. Add `range(self)` &mdash; the difference between the warmest and coldest recorded day.
2. Add `warm_spell(self, threshold=30.0)` &mdash; the length of the **longest run** of
   consecutive days at or above `threshold`.
3. Create two stations, give them a couple of weeks of data, and print both quantities.

**Hints.**
* `range` just needs `max(self.temperature) - min(self.temperature)`.
* For the longest run: walk the list with a counter that resets to 0 whenever a day is
  below the threshold, and track the best value seen.
* A method is a function defined inside the `class` block, with `self` as its first
  argument.


In [ ]:
class Station:
    """A weather station and its record of daily mean temperature (from Lecture 3)."""

    def __init__(self, name, lat, lon):
        self.name = name
        self.lat = lat
        self.lon = lon
        self.temperature = []             # daily values [degC]

    def add_day(self, value):
        self.temperature.append(value)

    def mean(self):
        return sum(self.temperature) / len(self.temperature)

    def hot_days(self, threshold=30.0):
        return sum(1 for t in self.temperature if t > threshold)

    # ==== YOUR CODE ====
    def range(self):
        return None                       # <-- replace

    def warm_spell(self, threshold=30.0):
        best = 0
        run = 0
        # for t in self.temperature:
        #     run = run + 1 if t >= threshold else 0
        #     best = max(best, run)
        return best                        # <-- complete the loop above
    # ===================


# --- test data (given) ---
mandi = Station("Mandi", 31.71, 76.93)
for t in [22, 24, 27, 31, 33, 34, 32, 29, 26, 28, 30, 31, 33, 35]:
    mandi.add_day(t)

delhi = Station("Delhi", 28.61, 77.21)
for t in [30, 32, 35, 38, 40, 41, 39, 37, 34, 36, 38, 40, 42, 41]:
    delhi.add_day(t)

for s in (mandi, delhi):
    print(f"{s.name:6s}  mean {s.mean():.1f}  range {s.range()}  "
          f"longest warm spell {s.warm_spell()} days")


```{admonition} Answers
:class: note
1. Which station has the larger temperature range, and does that match your expectation
   for a hill town versus a plains city?
2. `hot_days` counts days *above* 30; `warm_spell` counts the longest *consecutive* run
   at or above 30. Why can two stations have the same `hot_days` but very different
   `warm_spell`, and which matters more for heat stress (Module 5)?
```


## Exercise 2 &mdash; a synthetic temperature series with NumPy

The same loop-and-array skills you use here reappear when you time-step the energy
balance model (Lecture 4) and the catchment model (Lecture 7).

**Your task.**
1. Build a day-of-year axis `doy = np.arange(365)` and a synthetic daily mean
   temperature: an annual sine wave plus random noise (given formula).
2. Compute the **annual mean**, the **day of the warmest value**, and a **30-day
   running mean**.
3. The given plot draws the series, the running mean, and your warmest day.

**Hints.**
* `temp = 18 + 9*np.sin(2*np.pi*(doy-110)/365) + rng.normal(0, 2.5, 365)` (a monsoon-ish
  peak around late April, day ~110+90).
* Warmest day: `np.argmax(temp)`.
* 30-day running mean: `np.convolve(temp, np.ones(30)/30, mode="same")`.


In [ ]:
rng = np.random.default_rng(0)
doy = np.arange(365)
temp = 18 + 9 * np.sin(2 * np.pi * (doy - 110) / 365) + rng.normal(0, 2.5, 365)

# ==== YOUR CODE ====
annual_mean = None            # <-- temp.mean()
warmest_doy = None            # <-- np.argmax(temp)
run30 = None                  # <-- np.convolve(temp, np.ones(30)/30, mode="same")
# ===================

print(f"annual mean   : {annual_mean}")
print(f"warmest day   : {warmest_doy}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(doy, temp, lw=0.8, color="0.6", label="daily")
if run30 is not None:
    ax.plot(doy, run30, lw=2, color="tab:red", label="30-day running mean")
if warmest_doy is not None:
    ax.plot(warmest_doy, temp[warmest_doy], "ko", label=f"warmest day ({warmest_doy})")
if annual_mean is not None:
    ax.axhline(annual_mean, color="tab:blue", ls="--", lw=1, label=f"annual mean {annual_mean:.1f}")
ax.set_xlabel("day of year"); ax.set_ylabel("temperature [degC]")
ax.set_title("Synthetic daily temperature"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. Is the warmest *single day* the same as the peak of the running mean? Why does
   smoothing move (and lower) the peak?
2. `mode="same"` keeps the output the same length as the input but distorts the first
   and last 15 days. Why &mdash; and how would you handle the edges properly?
```


## Exercise 3 &mdash; a seasonal cycle from gridded data with xarray

This is the workflow of Module 4: open a NetCDF dataset, select a location, and reduce
it in time.

**Your task.**
1. Open the sample air-temperature dataset (given) and convert it to °C.
2. Select the grid point **nearest 40°N, 260°E** with `.sel(..., method="nearest")`.
3. Compute the **mean seasonal cycle** with
   `.groupby("time.month").mean()` and plot it (plot given).
4. Also compute and print the month-to-month **range** (warmest month minus coldest).

**Hints.**
* `air = xr.tutorial.open_dataset("air_temperature").air - 273.15` (downloads ~4 MB the
  first time; needs internet, which Colab has).
* `point = air.sel(lat=40, lon=260, method="nearest")`.
* `seasonal = point.groupby("time.month").mean()` &mdash; a 12-value DataArray indexed
  by `month`.


In [ ]:
import xarray as xr

air = xr.tutorial.open_dataset("air_temperature").air - 273.15    # [degC]
print(air)

# ==== YOUR CODE ====
point = None          # <-- air.sel(lat=40, lon=260, method="nearest")
seasonal = None       # <-- point.groupby("time.month").mean()
season_range = None   # <-- float(seasonal.max() - seasonal.min())
# ===================

print(f"\nseasonal temperature range at this point: {season_range} degC")


In [ ]:
# --- plot (given) ---
if seasonal is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(seasonal["month"], seasonal, "o-")
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("month"); ax.set_ylabel("mean temperature [degC]")
    ax.set_title(f"Seasonal cycle near {float(point.lat):.0f}N, {float(point.lon):.0f}E")
    ax.grid(alpha=0.3)
    fig.tight_layout()
else:
    print("fill in the cell above first")


```{admonition} Answers
:class: note
1. In which month is this location warmest and coldest? Is the cycle symmetric about
   the solstices, or does it lag &mdash; and why would it lag?
2. Change the point to `lat=25, lon=280` (over water) and re-run. Is the seasonal range
   larger or smaller than over land? Explain using the heat-capacity idea from
   Lecture 4.
3. This dataset covers only North America. Name the dataset you would open instead to do
   the same analysis for a point in the Indian Ocean (hint: Exercise 9).
```


## Where this goes next

Every later notebook uses these three skills: a time-stepping `class`
(**Lectures 4&ndash;8**), NumPy array loops for finite-difference schemes
(**Lectures 9&ndash;11**), and xarray `.sel` / `.groupby` for real climate data
(**Exercise 9** and **Module 4**).

```{note} Sources
Adapted from the *Introduction to Python* exercises of the
[*Climate of the Ocean*](https://github.com/florianboergel/climateoftheocean) course,
for **CE524 Applied Hydroclimatology**. Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
